# WideBind — Colab Training (D=3584, T4)
165M params, 24 layers, grouped MLP, cognitive mirror, adaptive gates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'
print(f'Data path: {DRIVE}')
!ls -lh "$DRIVE"/*.bin 2>/dev/null || echo 'No .bin files found'

In [ ]:
# Locate source files on Drive and copy locally
import sys, os, shutil
SRC_LOCAL = '/content/widebind_src'

src_drive = None
for d in [os.path.join(DRIVE, 'src'), DRIVE]:
    subdirs = ['core', 'compression', 'scripts', 'notebooks']
    if all(os.path.isdir(os.path.join(d, sd)) for sd in subdirs):
        src_drive = d
        break

if src_drive:
    if os.path.isdir(SRC_LOCAL):
        shutil.rmtree(SRC_LOCAL)
    shutil.copytree(src_drive, SRC_LOCAL)
    sys.path.insert(0, SRC_LOCAL)
    sys.path.insert(0, os.path.join(SRC_LOCAL, 'scripts'))
    print(f'Source files copied from {src_drive}')
else:
    print('Directories not found on Drive. Checking files...')
    for d in [os.path.join(DRIVE, 'src'), DRIVE]:
        for sd in ['core', 'compression', 'scripts']:
            p = os.path.join(d, sd)
            exists = os.path.isdir(p)
            print(f'  {p}: {"OK" if exists else "MISSING"}')
    print('\nUpload directories: core/, compression/, scripts/, notebooks/ to Drive')

%cd /content

In [ ]:
# Verify imports
from core import WideBindConfig, WideBindStack
from compression import FCF_CPR

cfg = WideBindConfig(D=3584, n_layers=24, bottleneck=3584, bind_K=16,
                     mlp_groups=32, mlp_expand=8)
model = WideBindStack(cfg)
print(f'Model: {model.param_count():,} params')
del model

In [ ]:
# ─── Launch training ───
# Config
D            = 3584
N_LAYERS     = 24
BIND_K       = 16
MLP_GROUPS   = 32
MLP_EXPAND   = 8
SEQ_LEN      = 128
LR           = 3e-4
MAX_STEPS    = 300000
WARMUP       = 2000
SAVE_EVERY   = 5000
EVAL_EVERY   = 1000
LOG_EVERY    = 100
SCHEDULER    = 'mirror'  # 'mirror' or 'cosine'

print('=' * 60)
print(f'WideBind D={D} G={MLP_GROUPS}x{MLP_EXPAND}×')
print(f'  Layers: {N_LAYERS} | K={BIND_K} | L={SEQ_LEN}')
print(f'  LR={LR} | warmup={WARMUP} | steps={MAX_STEPS}')
print(f'  Scheduler: {SCHEDULER}')
print(f'  Save dir: {os.path.join(DRIVE, "checkpoints")}')
print('=' * 60)

In [ ]:
# ─── Training loop ───
import os, math, sys, time, glob
import torch
import torch.nn.functional as F
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {mem_total:.1f} GB')

# --- config ---
cfg = WideBindConfig(
    D=D, n_layers=N_LAYERS, bottleneck=D, bind_K=BIND_K,
    mlp_groups=MLP_GROUPS, mlp_expand=MLP_EXPAND,
    seq_len=SEQ_LEN, lr=LR, max_steps=MAX_STEPS,
    warmup_steps=WARMUP, scheduler=SCHEDULER,
    data_dir=DRIVE, save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    grad_clip=1.0, conv_kernel=48,
)

# --- data ---
print(f'Loading data from {DRIVE}')
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
if not stream_files:
    raise FileNotFoundError(f'No token_stream_*.bin in {DRIVE}')
streams = []
for f in stream_files:
    data = np.fromfile(f, dtype=np.uint16)
    streams.append(data)
total_tokens = sum(len(s) for s in streams)
print(f'Found {len(streams)} files, {total_tokens:,} total tokens')

# --- model ---
model = WideBindStack(cfg).to(device)
n_params = model.param_count()
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')

# --- batch size ---
print('Finding optimal batch size...')
for bs in [8, 6, 4, 2, 1]:
    torch.cuda.empty_cache()
    try:
        x = torch.randint(0, 50000, (bs, SEQ_LEN), device=device)
        h = model.embed_tokens(x)
        out, _ = model(h, None)
        out[:, :1].sum().backward()
        model.zero_grad()
        print(f'  Batch size {bs}: OK')
        cfg.batch_size = bs
        break
    except RuntimeError:
        model.zero_grad()
        torch.cuda.empty_cache()
        print(f'  Batch size {bs}: OOM')
print(f'  Using batch size: {cfg.batch_size}')

# --- optimizer ---
from core import MirrorLRScheduler, AdaptiveController
param_groups = model.param_groups()
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg.lr, warmup=cfg.warmup_steps)

# --- FCF-CPR ---
cpr = FCF_CPR()

# --- resume ---
start_step = 0
best_val_loss = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)

def step_key(p):
    base = os.path.basename(p).replace('step_', '').replace('_fcf', '').split('.')[0]
    return int(base)
ckpt_files = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')), key=step_key)
if ckpt_files:
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = cpr.load_compressed(latest, cfg=cfg)
    print(f'  Keys in checkpoint: {list(ckpt.keys())}')
    missing, unexpected = model.load_state_dict(ckpt['model'], strict=False)
    if missing: print(f'  Missing keys: {len(missing)}')
    if unexpected: print(f'  Unexpected keys: {len(unexpected)}')
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    else:
        print('  WARNING: no optimizer in checkpoint, starting fresh')
    if 'scheduler' in ckpt:
        try:
            scheduler.load_state_dict(ckpt['scheduler'])
        except Exception as e:
            print(f'  WARNING: scheduler load failed ({e}), using step')
            scheduler._step = ckpt.get('step', 0)
    else:
        scheduler._step = ckpt.get('step', 0)
    start_step = ckpt['step']
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resume at step {start_step}')

# --- helpers ---
@torch.no_grad()
def evaluate_model(model, streams, cfg, device):
    total_loss = 0.0
    total_steps = 0
    n_eval = min(100, sum(len(s) for s in streams) // (cfg.batch_size * cfg.seq_len) // len(streams))
    n_warmup = 10
    for stream in streams:
        offset = max(len(stream) // 4, cfg.batch_size * cfg.seq_len + 1)
        state = None
        # Warm-up: let state + controller stabilize before measuring
        for i in range(n_warmup):
            needed = cfg.batch_size * cfg.seq_len + 1
            if offset + needed > len(stream):
                offset = 0; break
            chunk = stream[offset:offset + needed]
            x = torch.from_numpy(chunk[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            offset += cfg.batch_size * cfg.seq_len
            h = model.embed_tokens(x)
            out, state = model(h, state)
        # Measure
        for i in range(n_eval):
            needed = cfg.batch_size * cfg.seq_len + 1
            if offset + needed > len(stream):
                offset = 0; break
            chunk = stream[offset:offset + needed]
            x = torch.from_numpy(chunk[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            y = torch.from_numpy(chunk[1:cfg.batch_size*cfg.seq_len+1].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            offset += cfg.batch_size * cfg.seq_len
            h = model.embed_tokens(x)
            out, state = model(h, state)
            loss = model.compute_loss(out, y)
            total_loss += loss.item()
            total_steps += 1
            if i % 50 == 49:
                state = None
    return total_loss / max(total_steps, 1)

# --- train loop ---
state = None
stream_idx = 0
offset = 0
tokens_seen = 0
t0 = time.time()

print(f'Training: step {start_step} → {MAX_STEPS}')
try:
    for step in range(start_step, MAX_STEPS):
        model.train()
        stream = streams[stream_idx]
        needed = cfg.batch_size * SEQ_LEN + 1
        if offset + needed > len(stream):
            offset = 0
            stream_idx = (stream_idx + 1) % len(streams)
            stream = streams[stream_idx]
            state = None
        chunk = stream[offset:offset + needed]
        x = torch.from_numpy(chunk[:cfg.batch_size*SEQ_LEN].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        y = torch.from_numpy(chunk[1:cfg.batch_size*SEQ_LEN+1].copy()).long().view(cfg.batch_size, SEQ_LEN).to(device)
        offset += cfg.batch_size * SEQ_LEN

        h = model.embed_tokens(x)
        out, state = model(h, state)
        loss = model.compute_loss(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        tokens_seen += cfg.batch_size * SEQ_LEN
        current_lr = scheduler.get_last_lr()[0]

        if step % LOG_EVERY == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            mem_u = torch.cuda.max_memory_allocated() / 1e9
            print(f'  step={step:>6} loss={loss.item():.4f} lr={current_lr:.2e} '
                  f'tok/s={tok_s:.0f} vram={mem_u:.1f}/{mem_total:.0f} GB')
            torch.cuda.reset_peak_memory_stats()

        if step > 0 and step % EVAL_EVERY == 0:
            val_loss = evaluate_model(model, streams, cfg, device)
            print(f'  EVAL step={step}: val_loss={val_loss:.4f} '
                  f'val_ppl={math.exp(val_loss):.2f}')
            torch.cuda.empty_cache()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                cpr.save_compressed({
                    'step': step, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_loss': best_val_loss, 'cfg': cfg,
                }, os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  New best! Saved to best.pt (compressed)')

        if step > 0 and step % SAVE_EVERY == 0:
            save_path = os.path.join(cfg.save_dir, f'step_{step}.pt')
            cpr.save_compressed({
                'step': step, 'model': model.state_dict(),
                'best_val_loss': best_val_loss, 'cfg': cfg,
            }, save_path)
            print(f'  Checkpoint saved: step_{step}.pt (compressed)')

except KeyboardInterrupt:
    save_path = os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt')
    cpr.save_compressed({
        'step': step, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg,
    }, save_path)
    print(f'\nInterrupted. Saved to {save_path} (compressed)')

In [ ]:
# ─── Analyze latest checkpoint ───
from scripts.analyze_checkpoint import generate_report
import glob, shutil

latest_ckpt = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')))
if latest_ckpt:
    report_path = generate_report(latest_ckpt[-1])
    # Copy report to Drive for persistence
    drive_report = os.path.join(cfg.log_dir, os.path.basename(report_path))
    os.makedirs(cfg.log_dir, exist_ok=True)
    shutil.copy2(report_path, drive_report)
    print(f'Report copied to {drive_report}')
    from IPython.display import HTML, display
    with open(report_path, 'r') as f:
        display(HTML(f.read()))
else:
    print('No checkpoints found')